In [19]:
# Đoạn code này tự động lấy Tên tài khoản và Tên máy tính
import getpass
import socket
user = getpass.getuser()
host = socket.gethostname()
def get_system_context():
    user = getpass.getuser()
    host = socket.gethostname()
    return f"[USER: {user}] [HOST: {host}]"

In [20]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

In [21]:
listings_df_raw = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("multiLine", "true") \
    .load("../data/Listings.csv")
    # .load("hdfs://master-node:9000/AirbnbData/Listings.csv")  # dùng khi master-node bật
listings_df_raw.show(truncate=False)

+----------+---------------------------------------------------+--------+----------+----------------------------+------------------+------------------+--------------------+-----------------+-------------------------+--------------------+----------------------+-------------------+--------+-----+--------+---------+----------------+------------+------------+--------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+--------------+--------------+--------------------+----------------------+-------------------------+---------------------+---------------------------+----------------------+-------------------+----------------+
|listing_id|na

In [22]:
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import ArrayType, StringType
listings_df = listings_df_raw.withColumn(
    "amenities",
    from_json(col("amenities"), ArrayType(StringType()))
)

In [23]:
from pyspark.sql.functions import when, col
# Quy đổi toàn bộ giá từ tiền tệ địa phương (Local Currency) sang USD
# Tỷ giá được cập nhật theo thời điểm hiện tại
listings_df_currency = listings_df.withColumn(
    "price_usd",
    when(col("city") == "New York", col("price") * 1.0)           # USD giữ nguyên
    .when(col("city") == "Paris", col("price") * 1.156)            # EUR -> USD
    .when(col("city") == "Rome", col("price") * 1.156)             # EUR -> USD
    .when(col("city") == "Sydney", col("price") * 0.7038)           # AUD -> USD
    .when(col("city") == "Hong Kong", col("price") * 0.1276)        # HKD -> USD
    .when(col("city") == "Bangkok", col("price") * 0.03048)         # THB -> USD
    .when(col("city") == "Mexico City", col("price") * 0.05795)     # MXN -> USD
    .when(col("city") == "Cape Town", col("price") * 0.06130)       # ZAR -> USD
    .when(col("city") == "Istanbul", col("price") *  0.02162)        # TRY -> USD
    .when(col("city") == "Rio de Janeiro", col("price") * 0.1961 )   # BRL -> USD
    .otherwise(col("price"))
)

In [24]:
listings_df_currency.show()

+----------+--------------------+--------+----------+--------------------+------------------+------------------+--------------------+-----------------+-------------------------+--------------------+----------------------+-------------------+--------+-----+--------+---------+----------------+------------+------------+--------+--------------------+-----+--------------+--------------+--------------------+----------------------+-------------------------+---------------------+---------------------------+----------------------+-------------------+----------------+------------------+
|listing_id|                name| host_id|host_since|       host_location|host_response_time|host_response_rate|host_acceptance_rate|host_is_superhost|host_total_listings_count|host_has_profile_pic|host_identity_verified|      neighbourhood|district| city|latitude|longitude|   property_type|   room_type|accommodates|bedrooms|           amenities|price|minimum_nights|maximum_nights|review_scores_rating|review_scor

In [25]:
listings_df_currency.printSchema()

root
 |-- listing_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- host_id: integer (nullable = true)
 |-- host_since: date (nullable = true)
 |-- host_location: string (nullable = true)
 |-- host_response_time: string (nullable = true)
 |-- host_response_rate: double (nullable = true)
 |-- host_acceptance_rate: double (nullable = true)
 |-- host_is_superhost: string (nullable = true)
 |-- host_total_listings_count: integer (nullable = true)
 |-- host_has_profile_pic: string (nullable = true)
 |-- host_identity_verified: string (nullable = true)
 |-- neighbourhood: string (nullable = true)
 |-- district: string (nullable = true)
 |-- city: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- property_type: string (nullable = true)
 |-- room_type: string (nullable = true)
 |-- accommodates: integer (nullable = true)
 |-- bedrooms: integer (nullable = true)
 |-- amenities: array (nullable = true)
 |    |-- el

In [26]:
reviews_df=spark.read.csv("../data/Reviews.csv",header=True,inferSchema=True)
# reviews_df=spark.read.csv("hdfs://master-node:9000/AirbnbData/Reviews.csv",header=True,inferSchema=True)  # dùng khi master-node bật
reviews_df.show()

+----------+---------+----------+-----------+
|listing_id|review_id|      date|reviewer_id|
+----------+---------+----------+-----------+
|     11798|330265172|2018-09-30|   11863072|
|     15383|330103585|2018-09-30|   39147453|
|     16455|329985788|2018-09-30|    1125378|
|     17919|330016899|2018-09-30|  172717984|
|     26827|329995638|2018-09-30|   17542859|
|     74561|330089224|2018-09-30|  173044789|
|    140355|330194958|2018-09-30|  160093807|
|    162163|329980859|2018-09-30|   94026758|
|    167998|329950677|2018-09-30|   35388162|
|    178188|330213008|2018-09-30|    3652511|
|    232956|330140226|2018-09-30|   57664145|
|    266725|330038354|2018-09-30|   77387165|
|    314288|330299075|2018-09-30|  192717587|
|    325880|330081449|2018-09-30|  154527568|
|    335003|329968377|2018-09-30|    3461699|
|    348747|330131287|2018-09-30|    9554201|
|    352851|330201364|2018-09-30|  142182690|
|    378714|330246144|2018-09-30|   15772951|
|    406852|330283854|2018-09-30| 

In [27]:
reviews_df.printSchema()

root
 |-- listing_id: integer (nullable = true)
 |-- review_id: integer (nullable = true)
 |-- date: date (nullable = true)
 |-- reviewer_id: integer (nullable = true)

root
 |-- listing_id: integer (nullable = true)
 |-- review_id: integer (nullable = true)
 |-- date: date (nullable = true)
 |-- reviewer_id: integer (nullable = true)



In [28]:
listings_df_currency.createOrReplaceTempView("listings")
reviews_df.createOrReplaceTempView("reviews")
spark.catalog.listTables()

[Table(name='listings', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True),
 Table(name='reviews', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True)]

[Table(name='listings', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True),
 Table(name='reviews', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True)]

In [29]:
# Minh chứng số dòng > 100.000 dòng
spark.sql("SELECT COUNT(*) AS Total_rows FROM listings").show()

+----------+
|Total_rows|
+----------+
|    279712|
+----------+

+----------+
|Total_rows|
+----------+
|    279712|
+----------+



In [30]:
# Minh chứng số cột >10
num_cols = len(listings_df.columns)
print(f"Tổng số cột: {num_cols}")

Tổng số cột: 33
Tổng số cột: 33


In [31]:
# Câu truy vấn 2: Đánh giá hiệu quả của danh hiệu Superhost đối với việc thu hút khách hàng và chất lượng dịch vụ
print(f"{get_system_context()} [INFO]: Query execution started...")
spark.sql("""SELECT
   CASE L.host_is_superhost
       WHEN 't' THEN 'superhost'
       ELSE 'none'
   END AS host_is_superhost,
   COUNT(DISTINCT L.listing_id) AS total_listings,
   COUNT(R.review_id) AS reviews,
   ROUND(AVG(L.host_response_rate)*100, 2) AS avg_host_reponse,
   ROUND(AVG(L.host_acceptance_rate)*100, 2) AS avg_host_acceptance,
   ROUND(AVG(L.review_scores_rating), 2) AS avg_rating,
   ROUND(AVG(L.review_scores_cleanliness), 2) AS avg_cleanliness,
   ROUND(AVG(L.review_scores_communication), 2) AS avg_communication,
   ROUND(AVG(L.review_scores_value), 2) AS avg_value
FROM listings L
JOIN reviews R ON L.listing_id = R.listing_id
WHERE L.host_is_superhost IS NOT NULL
GROUP BY L.host_is_superhost
ORDER BY L.host_is_superhost ASC;""").toPandas()

[USER: Quynh Nga] [HOST: DESKTOP-TC9PES0] [INFO]: Query execution started...


,host_is_superhost,total_listings,reviews,avg_host_reponse,avg_host_acceptance,avg_rating,avg_cleanliness,avg_communication,avg_value
0,none,147904,3009692,89.88,87.70,92.80,9.27,9.72,9.27
1,superhost,45540,2359512,97.24,93.37,96.85,9.79,9.97,9.73


[USER: Quynh Nga] [HOST: DESKTOP-TC9PES0] [INFO]: Query execution started...


,host_is_superhost,total_listings,reviews,avg_host_reponse,avg_host_acceptance,avg_rating,avg_cleanliness,avg_communication,avg_value
0,none,147904,3009692,89.88,87.70,92.80,9.27,9.72,9.27
1,superhost,45540,2359512,97.24,93.37,96.85,9.79,9.97,9.73


In [32]:
# Câu truy vấn 3: Cơ cấu nguồn cung phòng và giá thuê theo quy mô sức chứa khách tại từng thành phố
print(f"{get_system_context()} [INFO]: Query execution started...")
spark.sql("""SELECT
   city,
   CASE
       WHEN accommodates <= 2 THEN '1-2 khách (Cá nhân/Cặp đôi)'
       WHEN accommodates BETWEEN 3 AND 5 THEN '3-5 khách (Gia đình/Nhóm nhỏ)'
       ELSE 'Trên 5 khách (Đoàn du lịch)'
   END AS room_capacity_segment,
   COUNT(listing_id) AS total_rooms_listed,
   ROUND(AVG(price_usd), 2) AS avg_nightly_price
FROM listings
WHERE accommodates IS NOT NULL
GROUP BY city,
   CASE
       WHEN accommodates <= 2 THEN '1-2 khách (Cá nhân/Cặp đôi)'
       WHEN accommodates BETWEEN 3 AND 5 THEN '3-5 khách (Gia đình/Nhóm nhỏ)'
       ELSE 'Trên 5 khách (Đoàn du lịch)'
   END
ORDER BY city,room_capacity_segment ASC;""").toPandas()

[USER: Quynh Nga] [HOST: DESKTOP-TC9PES0] [INFO]: Query execution started...


,city,room_capacity_segment,total_rooms_listed,avg_nightly_price
0,Bangkok,1-2 khách (Cá nhân/Cặp đôi),10976,48.26
1,Bangkok,3-5 khách (Gia đình/Nhóm nhỏ),6288,64.54
2,Bangkok,Trên 5 khách (Đoàn du lịch),2097,138.71
3,Cape Town,1-2 khách (Cá nhân/Cặp đôi),8769,60.74
4,Cape Town,3-5 khách (Gia đình/Nhóm nhỏ),5814,97.71
5,Cape Town,Trên 5 khách (Đoàn du lịch),4503,380.47
6,Hong Kong,1-2 khách (Cá nhân/Cặp đôi),4912,69.18
7,Hong Kong,3-5 khách (Gia đình/Nhóm nhỏ),1486,132.99
8,Hong Kong,Trên 5 khách (Đoàn du lịch),689,199.31
9,Istanbul,1-2 khách (Cá nhân/Cặp đôi),12417,8.47


[USER: Quynh Nga] [HOST: DESKTOP-TC9PES0] [INFO]: Query execution started...


,city,room_capacity_segment,total_rooms_listed,avg_nightly_price
0,Bangkok,1-2 khách (Cá nhân/Cặp đôi),10976,48.26
1,Bangkok,3-5 khách (Gia đình/Nhóm nhỏ),6288,64.54
2,Bangkok,Trên 5 khách (Đoàn du lịch),2097,138.71
3,Cape Town,1-2 khách (Cá nhân/Cặp đôi),8769,60.74
4,Cape Town,3-5 khách (Gia đình/Nhóm nhỏ),5814,97.71
5,Cape Town,Trên 5 khách (Đoàn du lịch),4503,380.47
6,Hong Kong,1-2 khách (Cá nhân/Cặp đôi),4912,69.18
7,Hong Kong,3-5 khách (Gia đình/Nhóm nhỏ),1486,132.99
8,Hong Kong,Trên 5 khách (Đoàn du lịch),689,199.31
9,Istanbul,1-2 khách (Cá nhân/Cặp đôi),12417,8.47


In [33]:
# Câu truy vấn 7: Đánh giá các tiện ích cốt lõi thuộc phân khúc căn hộ Cao cấp
print(f"{get_system_context()} [INFO]: Query execution started...")
spark.sql("""SELECT
   exploded_amenity AS amenity_name,
   COUNT(*) AS total_occurrence
FROM listings
LATERAL VIEW EXPLODE(amenities) AS exploded_amenity
WHERE price_usd > (SELECT AVG(price_usd) FROM listings)
GROUP BY exploded_amenity
ORDER BY total_occurrence DESC
LIMIT 5;""").toPandas()

[USER: Quynh Nga] [HOST: DESKTOP-TC9PES0] [INFO]: Query execution started...


,amenity_name,total_occurrence
0,Wifi,66128
1,Essentials,62925
2,Kitchen,62348
3,Long term stays allowed,60150
4,TV,59949


[USER: Quynh Nga] [HOST: DESKTOP-TC9PES0] [INFO]: Query execution started...


,amenity_name,total_occurrence
0,Wifi,66128
1,Essentials,62925
2,Kitchen,62348
3,Long term stays allowed,60150
4,TV,59949


In [34]:
# QUERY 1 – Diem hanh vi tot & ty le Superhost
spark.sql("""
  SELECT diem_hanh_vi_tot,
         COUNT(*)                       AS so_listing,
         ROUND(100.0*AVG(la_super), 2)  AS ty_le_superhost
  FROM (
      SELECT CASE WHEN host_is_superhost = 't' THEN 1 ELSE 0 END AS la_super,
             ( CASE WHEN host_response_time   = 'within an hour' THEN 1 ELSE 0 END
             + CASE WHEN host_identity_verified = 't'            THEN 1 ELSE 0 END
             + CASE WHEN instant_bookable       = 't'            THEN 1 ELSE 0 END ) AS diem_hanh_vi_tot
      FROM listings
  )
  GROUP BY diem_hanh_vi_tot
  ORDER BY diem_hanh_vi_tot
""").show()

+----------------+----------+---------------+
|diem_hanh_vi_tot|so_listing|ty_le_superhost|
+----------------+----------+---------------+
|               0|     40076|           5.97|
|               1|    118466|          12.59|
|               2|     81714|          23.74|
|               3|     39456|          34.33|
+----------------+----------+---------------+

+----------------+----------+---------------+
|diem_hanh_vi_tot|so_listing|ty_le_superhost|
+----------------+----------+---------------+
|               0|     40076|           5.97|
|               1|    118466|          12.59|
|               2|     81714|          23.74|
|               3|     39456|          34.33|
+----------------+----------+---------------+



In [17]:
# QUERY 9 – Top 5 Superhost rui ro nhat trong tung nhom
spark.sql("""
WITH host_agg AS (
 SELECT host_id, FIRST(city) AS city, COUNT(*) AS n_listing,
        ROUND(AVG(review_scores_rating),2)  AS rating_tb,
        MAX(host_is_superhost) AS is_super
 FROM listings
 WHERE review_scores_rating IS NOT NULL
 GROUP BY host_id
 HAVING MAX(host_is_superhost) = 't'
),
city_bench AS (
 SELECT city, AVG(rating_tb) AS rating_city FROM host_agg GROUP BY city
),
RankedAnalysis AS (
 SELECT h.host_id, h.city, h.n_listing, h.rating_tb,
        ROUND(h.rating_tb - c.rating_city, 2) AS lech_vs_city,
        NTILE(4) OVER (ORDER BY h.rating_tb) AS nhom_rui_ro,
        CASE WHEN h.rating_tb < c.rating_city THEN 'SUPERHOST RUI RO'
             ELSE 'OK' END AS canh_bao
 FROM host_agg h JOIN city_bench c ON h.city = c.city
 WHERE h.n_listing >= 2
)
SELECT host_id, city, n_listing, rating_tb, lech_vs_city, nhom_rui_ro, canh_bao
FROM (
 SELECT *,
        ROW_NUMBER() OVER (PARTITION BY nhom_rui_ro ORDER BY lech_vs_city ASC) AS thut_tu_trong_nhom
 FROM RankedAnalysis
)
WHERE thut_tu_trong_nhom <= 5
ORDER BY nhom_rui_ro ASC, lech_vs_city ASC
""").show(20)

+---------+--------------+---------+---------+------------+-----------+----------------+
|  host_id|          city|n_listing|rating_tb|lech_vs_city|nhom_rui_ro|        canh_bao|
+---------+--------------+---------+---------+------------+-----------+----------------+
|309651408|     Cape Town|        2|     50.0|      -47.68|          1|SUPERHOST RUI RO|
|125422956|        Sydney|        2|     58.5|      -39.16|          1|SUPERHOST RUI RO|
| 31168193|     Cape Town|        2|     70.0|      -27.68|          1|SUPERHOST RUI RO|
|152690866|      Istanbul|        5|     71.0|      -26.28|          1|SUPERHOST RUI RO|
|274418925|       Bangkok|        3|    71.67|       -25.4|          1|SUPERHOST RUI RO|
|278252599|   Mexico City|        2|     96.0|       -1.68|          2|SUPERHOST RUI RO|
|271659755|   Mexico City|        2|     96.0|       -1.68|          2|SUPERHOST RUI RO|
|239389564|   Mexico City|        3|     96.0|       -1.68|          2|SUPERHOST RUI RO|
|211773688|   Mexico 

In [18]:
# QUERY 10 – PIVOT: Ty le Superhost theo loai phong & thanh pho
spark.sql("""
SELECT * FROM (
 SELECT city, room_type,
        CASE WHEN host_is_superhost='t' THEN 1.0 ELSE 0.0 END AS is_super
 FROM listings
) PIVOT (
 ROUND(AVG(is_super)*100, 1)
 FOR room_type IN ('Entire place' AS entire, 'Private room' AS private,
                   'Hotel room' AS hotel, 'Shared room' AS shared)
)
""").show()


+--------------+------+-------+-----+------+
|          city|entire|private|hotel|shared|
+--------------+------+-------+-----+------+
|        Sydney|  13.8|    9.7| 22.7|   2.5|
|   Mexico City|  39.8|   23.1| 25.2|  18.9|
|     Cape Town|  27.6|   14.5| 11.4|   7.1|
|         Paris|  12.0|   15.7| 20.3|   8.5|
|      Istanbul|  18.0|    7.8| 14.8|   6.0|
|     Hong Kong|  14.6|   21.7|  3.9|  13.7|
|          Rome|  30.4|   18.4| 18.4|  10.0|
|Rio de Janeiro|  18.7|   13.1| 26.3|   8.7|
|       Bangkok|  25.7|   12.6| 16.1|  15.0|
|      New York|  20.0|   17.6| 14.4|  15.7|
+--------------+------+-------+-----+------+

